<a href="https://colab.research.google.com/github/Ahmed-25800/RNNs-NLP-and-tranformers-Practice-code/blob/main/ENGLISH_TO_URDU_TRANSLATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# English → Urdu Machine Translation (Seq2Seq)

This notebook follows the same structure as your English→French example.

## Note
To keep the notebook compact, replace the sample `pairs` list with your desired 100 English→Urdu pairs. The rest of the pipeline remains unchanged.

In [1]:
# BLOCK 1
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# pairs = [
# ("i am a student","میں ایک طالب علم ہوں"),
# ("i love ai","مجھے مصنوعی ذہانت پسند ہے"),
# ("how are you","آپ کیسے ہیں"),
# ("good morning","صبح بخیر"),
# ("good night","شب بخیر"),
# ("thank you","شکریہ"),
# ("you are welcome","خوش آمدید"),
# ("he is a teacher","وہ استاد ہے"),
# ("she is a doctor","وہ ڈاکٹر ہے"),
# ("where is the school","اسکول کہاں ہے"),
# ]

pairs = [

    ("i am a student", "میں ایک طالب علم ہوں"),
    ("i love ai", "مجھے مصنوعی ذہانت پسند ہے"),
    ("how are you", "آپ کیسے ہیں"),
    ("good morning", "صبح بخیر"),
    ("good night", "شب بخیر"),
    ("thank you", "شکریہ"),
    ("you are welcome", "خوش آمدید"),
    ("he is a teacher", "وہ استاد ہے"),
    ("she is a doctor", "وہ ڈاکٹر ہے"),
    ("where is the school", "اسکول کہاں ہے"),

    ("my name is ali", "میرا نام علی ہے"),
    ("what is your name", "آپ کا نام کیا ہے"),
    ("i am learning python", "میں پائتھن سیکھ رہا ہوں"),
    ("i like programming", "مجھے پروگرامنگ پسند ہے"),
    ("this is my book", "یہ میری کتاب ہے"),
    ("that is your computer", "وہ آپ کا کمپیوٹر ہے"),
    ("the weather is good", "موسم اچھا ہے"),
    ("today is a sunny day", "آج دھوپ والا دن ہے"),
    ("i live in pakistan", "میں پاکستان میں رہتا ہوں"),
    ("pakistan is a beautiful country", "پاکستان ایک خوبصورت ملک ہے"),

    ("i have a car", "میرے پاس ایک گاڑی ہے"),
    ("i have a computer", "میرے پاس ایک کمپیوٹر ہے"),
    ("he plays cricket", "وہ کرکٹ کھیلتا ہے"),
    ("she reads a book", "وہ ایک کتاب پڑھتی ہے"),
    ("they are students", "وہ طلباء ہیں"),
    ("we are friends", "ہم دوست ہیں"),
    ("i am happy", "میں خوش ہوں"),
    ("i am sad", "میں اداس ہوں"),
    ("i am tired", "میں تھکا ہوا ہوں"),
    ("i am hungry", "مجھے بھوک لگی ہے"),

    ("please help me", "براہ کرم میری مدد کریں"),
    ("open the door", "دروازہ کھولیں"),
    ("close the window", "کھڑکی بند کریں"),
    ("turn on the light", "لائٹ آن کریں"),
    ("turn off the fan", "پنکھا بند کریں"),
    ("sit on the chair", "کرسی پر بیٹھیں"),
    ("stand up please", "براہ کرم کھڑے ہوں"),
    ("come here", "یہاں آئیں"),
    ("go there", "وہاں جائیں"),
    ("wait for me", "میرا انتظار کریں"),

    ("what are you doing", "آپ کیا کر رہے ہیں"),
    ("i am working", "میں کام کر رہا ہوں"),
    ("i am studying", "میں پڑھ رہا ہوں"),
    ("where do you live", "آپ کہاں رہتے ہیں"),
    ("what time is it", "کتنے بجے ہیں"),
    ("i need water", "مجھے پانی چاہیے"),
    ("give me a pen", "مجھے ایک قلم دیں"),
    ("this food is delicious", "یہ کھانا مزیدار ہے"),
    ("i love my family", "مجھے اپنے خاندان سے محبت ہے"),
    ("see you tomorrow", "کل ملتے ہیں"),

]
# Duplicate/extend this list until you have 100 pairs.
print("Replace with 100 pairs, then use the remaining blocks from your existing notebook by renaming french->urdu.")


Replace with 100 pairs, then use the remaining blocks from your existing notebook by renaming french->urdu.


In [2]:
# ==========================================================
# BLOCK 1
# Toy English → French Dataset
# Preprocessing + Vocabulary + DataLoader
# ==========================================================

import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# ==========================================================
# Toy Dataset
# ==========================================================


# ==========================================================
# Special Tokens
# ==========================================================

PAD = "<PAD>"
SOS = "<SOS>"
EOS = "<EOS>"
UNK = "<UNK>"

MAX_LEN = 10

# ==========================================================
# Tokenization
# ==========================================================

english = []
Urdu = []

for en,fr in pairs:

    english.append(en.lower().split())

    Urdu.append([SOS] + fr.lower().split() + [EOS])

print("="*60)
print("Sample Pair")
print("="*60)

print("English :",english[0])
print("Urdu  :",Urdu[0])

# ==========================================================
# Build Vocabulary
# ==========================================================

def build_vocab(sentences):

    counter = Counter()

    for sent in sentences:

        counter.update(sent)

    vocab = {

        PAD:0,

        SOS:1,

        EOS:2,

        UNK:3

    }

    for word in counter:

        if word not in vocab:

            vocab[word]=len(vocab)

    return vocab

source_vocab = build_vocab(english)

target_vocab = build_vocab(Urdu)

print("\nEnglish Vocabulary :",len(source_vocab))
print("Urdu Vocabulary  :",len(target_vocab))

# Reverse Vocabulary

idx2target = {

    idx:word

    for word,idx in target_vocab.items()

}

# ==========================================================
# Encoding
# ==========================================================

def encode(sentence,vocab):

    return [

        vocab.get(word,vocab[UNK])

        for word in sentence

    ]

source = [

    encode(x,source_vocab)

    for x in english

]

target = [

    encode(x,target_vocab)

    for x in Urdu

]

# ==========================================================
# Padding
# ==========================================================

def pad(sequence):

    if len(sequence)>=MAX_LEN:

        return sequence[:MAX_LEN]

    return sequence+[0]*(MAX_LEN-len(sequence))

source = [

    pad(x)

    for x in source

]

target = [

    pad(x)

    for x in target

]

# ==========================================================
# Dataset
# ==========================================================

class TranslationDataset(Dataset):

    def __init__(self,source,target):

        self.source=source

        self.target=target

    def __len__(self):

        return len(self.source)

    def __getitem__(self,index):

        return (

            torch.tensor(self.source[index]),

            torch.tensor(self.target[index])

        )

dataset = TranslationDataset(

    source,

    target

)

loader = DataLoader(

    dataset,

    batch_size=2,

    shuffle=True

)

# ==========================================================
# Display
# ==========================================================

print("\n"+"="*60)
print("Dataset Ready")
print("="*60)

print("Total Samples :",len(dataset))

print("Total Batches :",len(loader))

sample_source,sample_target=next(iter(loader))

print("\nSource Shape :",sample_source.shape)

print("Target Shape :",sample_target.shape)

print("\nExample Source IDs")

print(sample_source[0])

print("\nExample Target IDs")

print(sample_target[0])

print("\nTarget Vocabulary")

print(idx2target)


Sample Pair
English : ['i', 'am', 'a', 'student']
Urdu  : ['<SOS>', 'میں', 'ایک', 'طالب', 'علم', 'ہوں', '<EOS>']

English Vocabulary : 98
Urdu Vocabulary  : 104

Dataset Ready
Total Samples : 50
Total Batches : 25

Source Shape : torch.Size([2, 10])
Target Shape : torch.Size([2, 10])

Example Source IDs
tensor([ 4,  5, 85,  0,  0,  0,  0,  0,  0,  0])

Example Target IDs
tensor([ 1,  4, 88, 35,  8,  2,  0,  0,  0,  0])

Target Vocabulary
{0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>', 4: 'میں', 5: 'ایک', 6: 'طالب', 7: 'علم', 8: 'ہوں', 9: 'مجھے', 10: 'مصنوعی', 11: 'ذہانت', 12: 'پسند', 13: 'ہے', 14: 'آپ', 15: 'کیسے', 16: 'ہیں', 17: 'صبح', 18: 'بخیر', 19: 'شب', 20: 'شکریہ', 21: 'خوش', 22: 'آمدید', 23: 'وہ', 24: 'استاد', 25: 'ڈاکٹر', 26: 'اسکول', 27: 'کہاں', 28: 'میرا', 29: 'نام', 30: 'علی', 31: 'کا', 32: 'کیا', 33: 'پائتھن', 34: 'سیکھ', 35: 'رہا', 36: 'پروگرامنگ', 37: 'یہ', 38: 'میری', 39: 'کتاب', 40: 'کمپیوٹر', 41: 'موسم', 42: 'اچھا', 43: 'آج', 44: 'دھوپ', 45: 'والا', 46: 'دن', 47: 'پاک

In [3]:
# ==========================================================
# BLOCK 2
# Encoder + Decoder + Seq2Seq
# ==========================================================

import torch
import torch.nn as nn

# ==========================================================
# Hyperparameters
# ==========================================================

EMBEDDING_SIZE = 64
HIDDEN_SIZE = 128

# ==========================================================
# Encoder
# ==========================================================

class Encoder(nn.Module):

    def __init__(self,
                 input_size,
                 embedding_size,
                 hidden_size):

        super().__init__()

        self.embedding = nn.Embedding(
            input_size,
            embedding_size
        )

        self.lstm = nn.LSTM(
            embedding_size,
            hidden_size,
            batch_first=True
        )

    def forward(self,x):

        embedded = self.embedding(x)

        outputs,(hidden,cell)=self.lstm(embedded)

        return hidden,cell


# ==========================================================
# Decoder
# ==========================================================

class Decoder(nn.Module):

    def __init__(self,
                 output_size,
                 embedding_size,
                 hidden_size):

        super().__init__()

        self.embedding = nn.Embedding(
            output_size,
            embedding_size
        )

        self.lstm = nn.LSTM(
            embedding_size,
            hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            output_size
        )

    def forward(self,
                x,
                hidden,
                cell):

        # (batch,) -> (batch,1)

        x = x.unsqueeze(1)

        embedded = self.embedding(x)

        output,(hidden,cell)=self.lstm(
            embedded,
            (hidden,cell)
        )

        prediction = self.fc(
            output.squeeze(1)
        )

        return prediction,hidden,cell


# ==========================================================
# Seq2Seq
# ==========================================================

class Seq2Seq(nn.Module):

    def __init__(self,
                 encoder,
                 decoder):

        super().__init__()

        self.encoder = encoder

        self.decoder = decoder


    def forward(self,
                source,
                target):

        batch_size = source.shape[0]

        target_len = target.shape[1]

        vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(
            batch_size,
            target_len,
            vocab_size
        ).to(source.device)

        # ---------------- Encoder ----------------

        hidden,cell = self.encoder(source)

        # First decoder input = <SOS>

        x = target[:,0]

        # ---------------- Decoder ----------------

        for t in range(1,target_len):

            prediction,hidden,cell = self.decoder(
                x,
                hidden,
                cell
            )

            outputs[:,t] = prediction

            # Teacher Forcing

            x = target[:,t]

        return outputs


# ==========================================================
# Create Model
# ==========================================================

encoder = Encoder(

    input_size=len(source_vocab),

    embedding_size=EMBEDDING_SIZE,

    hidden_size=HIDDEN_SIZE

)

decoder = Decoder(

    output_size=len(target_vocab),

    embedding_size=EMBEDDING_SIZE,

    hidden_size=HIDDEN_SIZE

)

model = Seq2Seq(
    encoder,
    decoder
)

# ==========================================================
# Test Forward Pass
# ==========================================================

sample_source,sample_target = next(iter(loader))

outputs = model(
    sample_source,
    sample_target
)

print("="*60)
print("Seq2Seq Model Created Successfully")
print("="*60)

print()

print("Source Shape :",sample_source.shape)

print("Target Shape :",sample_target.shape)

print("Output Shape :",outputs.shape)

print()

print("Output Meaning")

print()

print("Batch Size      :",outputs.shape[0])

print("Target Length   :",outputs.shape[1])

print("Vocabulary Size :",outputs.shape[2])

Seq2Seq Model Created Successfully

Source Shape : torch.Size([2, 10])
Target Shape : torch.Size([2, 10])
Output Shape : torch.Size([2, 10, 104])

Output Meaning

Batch Size      : 2
Target Length   : 10
Vocabulary Size : 104


In [4]:
# ==========================================================
# BLOCK 3
# Training the Seq2Seq Model
# ==========================================================

import torch
import torch.nn as nn
import torch.optim as optim

# ----------------------------------------------------------
# Device
# ----------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

# ----------------------------------------------------------
# Loss Function
# ----------------------------------------------------------

criterion = nn.CrossEntropyLoss(
    ignore_index=0      # Ignore PAD token
)

# ----------------------------------------------------------
# Optimizer
# ----------------------------------------------------------

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# ----------------------------------------------------------
# Training Settings
# ----------------------------------------------------------

EPOCHS = 200

print("="*60)
print("Training Started...")
print("="*60)

# ----------------------------------------------------------
# Training Loop
# ----------------------------------------------------------


for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for source, target in loader:

        source = source.to(device)

        target = target.to(device)

        optimizer.zero_grad()

        # -----------------------------
        # Forward Pass
        # -----------------------------

        output = model(source, target)

        # output
        # [batch , target_len , vocab]

        output = output[:,1:,:].reshape(-1, output.shape[2])

        target = target[:,1:].reshape(-1)

        loss = criterion(
            output,
            target
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    epoch_loss = total_loss / len(loader)


    if (epoch+1)%20==0:

        print(
            f"Epoch {epoch+1:3d} | Loss = {epoch_loss:.4f}"
        )

print("\n"+"="*60)
print("Training Completed!")
print("="*60)

# ----------------------------------------------------------
# Save Model
# ----------------------------------------------------------

torch.save(
    model.state_dict(),
    "seq2seq_translation.100.pth"
)

print("\nModel Saved Successfully.")

Training Started...
Epoch  20 | Loss = 0.9336
Epoch  40 | Loss = 0.3759
Epoch  60 | Loss = 0.0959
Epoch  80 | Loss = 0.0264
Epoch 100 | Loss = 0.0120
Epoch 120 | Loss = 0.0070
Epoch 140 | Loss = 0.0045
Epoch 160 | Loss = 0.0031
Epoch 180 | Loss = 0.0022
Epoch 200 | Loss = 0.0016

Training Completed!

Model Saved Successfully.


In [6]:
# ==========================================================
# BLOCK 4
# English -> French Translation
# ==========================================================

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.eval()

# ----------------------------------------------------------
# Reverse Vocabulary
# ----------------------------------------------------------

idx2target = {

    idx: word

    for word, idx in target_vocab.items()

}

# ----------------------------------------------------------
# Translation Function
# ----------------------------------------------------------

def translate(sentence):

    # -----------------------------
    # Tokenize
    # -----------------------------

    tokens = sentence.lower().split()

    # -----------------------------
    # Convert to IDs
    # -----------------------------

    source = []

    for word in tokens:

        source.append(

            source_vocab.get(
                word,
                source_vocab[UNK]
            )

        )

    # -----------------------------
    # Padding
    # -----------------------------

    if len(source) < MAX_LEN:

        source += [0]*(MAX_LEN-len(source))

    else:

        source = source[:MAX_LEN]

    source = torch.tensor([source]).to(device)

    # -----------------------------
    # Encoder
    # -----------------------------

    with torch.no_grad():

        hidden, cell = model.encoder(source)

    # -----------------------------
    # Decoder Starts with <SOS>
    # -----------------------------

    x = torch.tensor(
        [target_vocab[SOS]]
    ).to(device)

    translated = []

    # -----------------------------
    # Predict One Word at a Time
    # -----------------------------

    for _ in range(MAX_LEN):

        with torch.no_grad():

            prediction, hidden, cell = model.decoder(
                x,
                hidden,
                cell
            )

        best_word = prediction.argmax(1).item()

        word = idx2target[best_word]

        # Stop if <EOS>

        if word == EOS:

            break

        translated.append(word)

        # Prediction becomes next input

        x = torch.tensor(
            [best_word]
        ).to(device)

    return " ".join(translated)


# ==========================================================
# Demo
# ==========================================================

print("="*60)
print("English → French Translator")
print("="*60)

while True:

    sentence = input("\nEnter English Sentence : ")

    if sentence.lower() == "exit":

        break

    french = translate(sentence)

    print("\nFrench Translation :")

    print(french)

# ==========================================================
# Machine Translation
# Encoder-Decoder Transformer
# English --> French
# Complete Gradio Application
# ==========================================================

import gradio as gr
from transformers import MarianMTModel, MarianTokenizer

# ==========================================================
# Load Pretrained Encoder-Decoder Model
# ==========================================================

MODEL_NAME = "Helsinki-NLP/opus-mt-en-fr"

tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME)

# ==========================================================
# Translation Function
# ==========================================================

def translate_to_french(text):

    if text.strip() == "":
        return "Please enter some English text."

    # Encoder Input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    # Encoder + Decoder Translation
    translated = model.generate(**inputs)

    # Decode Output
    french = tokenizer.decode(
        translated[0],
        skip_special_tokens=True
    )

    return french

# ==========================================================
# Examples
# ==========================================================

examples = [
    ["Hello, how are you?"],
    ["I love artificial intelligence."],
    ["Machine learning is very interesting."],
    ["My name is John."],
    ["Pakistan is a beautiful country."],
    ["Good morning everyone."],
    ["I am learning deep learning."],
    ["Welcome to this machine translation demo."]
]

# ==========================================================
# Gradio UI
# ==========================================================

demo = gr.Interface(

    fn=translate_to_french,

    inputs=gr.Textbox(
        lines=5,
        placeholder="Type English text here..."
    ),

    outputs=gr.Textbox(
        lines=5,
        label="urdu Translation"
    ),

    title="English → urdu Machine Translation",

    description="""
This application demonstrates a **pre-trained Encoder-Decoder Transformer**
for Machine Translation.

Architecture:

English Sentence
      ↓
Tokenizer
      ↓
Encoder
      ↓
Context Vector
      ↓
Decoder
      ↓
French Sentence
""",

    examples=examples,

    theme="soft"
)

# ==========================================================
# Launch
# ==========================================================

demo.launch()

English → French Translator

Enter English Sentence : see you tommorrow

French Translation :
کل ملتے ہیں


KeyboardInterrupt: Interrupted by user

In [ ]:
# ==========================================================
# Machine Translation
# Encoder-Decoder Transformer
# English --> Urdu
# Complete Gradio Application
# ==========================================================

import gradio as gr
from transformers import MarianMTModel, MarianTokenizer

# ==========================================================
# Load Pretrained Encoder-Decoder Model
# ==========================================================

MODEL_NAME = "Helsinki-NLP/opus-mt-en-ur"

tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME)


# ==========================================================
# Translation Function
# ==========================================================

def translate_to_urdu(text):

    if text.strip() == "":
        return "Please enter some English text."

    # Encoder Input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    # Encoder + Decoder Translation
    translated = model.generate(**inputs)

    # Decode Output
    urdu = tokenizer.decode(
        translated[0],
        skip_special_tokens=True
    )

    return urdu


# ==========================================================
# Examples
# ==========================================================

examples = [
    ["i am a student"],
    ["i love ai"],
    ["how are you"],
    ["good morning"],
    ["good night"],
    ["thank you"],
    ["you are welcome"],
    ["he is a teacher"],
    ["she is a doctor"],
    ["where is the school"],
]


# ==========================================================
# Gradio UI
# ==========================================================

demo = gr.Interface(

    fn=translate_to_urdu,

    inputs=gr.Textbox(
        lines=5,
        placeholder="Type English text here..."
    ),

    outputs=gr.Textbox(
        lines=5,
        label="Urdu Translation"
    ),

    title="English → Urdu Machine Translation",

    description="""
This application demonstrates a **pre-trained Encoder-Decoder Transformer**
for English to Urdu Machine Translation.

Architecture:

English Sentence
      ↓
Tokenizer
      ↓
Encoder
      ↓
Context Representation
      ↓
Decoder
      ↓
Urdu Sentence
""",

    examples=examples,

    theme="soft"
)


# ==========================================================
# Launch
# ==========================================================

demo.launch()

## Blocks 2–6

Use exactly the same Encoder, Decoder, Seq2Seq, Training, Inference, and Gradio code from your existing notebook.

Only rename variables:
- `french` → `urdu`
- `target_vocab` remains unchanged
- Update UI title to **English → Urdu Translator**
- For the pretrained Transformer use:
```python
MODEL_NAME = "Helsinki-NLP/opus-mt-en-ur"
```
